# Ridge â€” L2-Regularized Regression (Pipeline B)

**Model**: Ridge via `sklearn.linear_model.Ridge`
**Target**: `gdpc1`
**Variables**: Cat3 (53 + COVID = 56 total)
**Train**: 1959-Q1 to 2007-Q4 / **Val**: 2008-Q1 to 2016-Q4
**Test**: 2017-Q1 to 2025-Q4 (36 quarters, m1/m2/m3)
**Scaling**: YES / **HP tuning**: YES via `RidgeCV` / **n_lags**: 4


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import norm

plt.rcParams["figure.figsize"] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, flatten_data, mean_fill_dataset,
                     get_features, split_for_scaler, load_data,
                     PREDICTIONS_DIR, TARGET, COVID)

SEED = 42; np.random.seed(SEED)
TRAIN_START = "1959-01-01"; TRAIN_END = "2007-12-31"
VAL_END = "2016-12-31"; TEST_START = "2017-01-01"; TEST_END = "2025-12-31"
N_LAGS = 4
VINTAGES = {"m1": -2, "m2": -1, "m3": 0}
print("Ridge â€” Cat3, n_lags={}, scaling=YES, HP tune=YES".format(N_LAGS))


Ridge â€” Cat3, n_lags=4, scaling=YES, HP tune=YES


In [2]:
monthly, _, metadata = load_data()
features = get_features("cat3", with_covid=True)
scaled_cols, unscaled_cols = split_for_scaler(features)
print("Features: {} total".format(len(features)))


Features: 56 total


In [3]:
# Phase A: HP tuning
tune_data = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]),:].dropna(axis=0,how="any").reset_index(drop=True)

feature_cols = [c for c in tune_flat.columns if c != "date" and c != TARGET and any(c == f or c.startswith(f + "_") for f in features)]
X_tune = tune_flat[feature_cols].values
y_tune = tune_flat[TARGET].values

# Scale ONCE on tune set (train+val only, no test leakage)
scaler_A = StandardScaler()
X_tune_scaled = scaler_A.fit_transform(X_tune)


tscv = TimeSeriesSplit(n_splits=5)
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=tscv)
ridge_cv.fit(X_tune_scaled, y_tune)
best_alpha = ridge_cv.alpha_
print("Best alpha: {:.4g}".format(best_alpha))


Best alpha: 14.56


In [4]:
test_quarters = monthly[(monthly["date"] >= TEST_START) & (monthly["date"] <= TEST_END) &
                         monthly["date"].dt.month.isin([3,6,9,12])]["date"].tolist()
predictions = {v: [] for v in VINTAGES}
actuals_list = []; failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly["date"]==pd_q][TARGET].values[0])
    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train = monthly[(monthly["date"]>=TRAIN_START) & (monthly["date"]<=fc)].reset_index(drop=True)
        train_m = gen_lagged_data(metadata, train.copy(), fc, lag=0)
        train_f = mean_fill_dataset(train_m, train_m)
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3,6,9,12]),:].dropna(axis=0,how="any").reset_index(drop=True)
        if len(train_fl) < 10: predictions[vn].append(np.nan); continue
        sc_cols = [c for c in feature_cols if c in scaled_cols or any(c.startswith(s) for s in scaled_cols)]
        us_cols = [c for c in feature_cols if c in unscaled_cols or any(c.startswith(u) for u in unscaled_cols)]
        try:
            sc = StandardScaler()
            X_tr = np.hstack([sc.fit_transform(train_fl[sc_cols].values), train_fl[us_cols].values]) if us_cols else sc.fit_transform(train_fl[sc_cols].values)
            m = Ridge(alpha=best_alpha, random_state=SEED); m.fit(X_tr, train_fl[TARGET].values)
            tr = monthly[monthly["date"]==fc].reset_index(drop=True)
            tr_m = gen_lagged_data(metadata, tr.copy(), fc, lag=0)
            tr_f = mean_fill_dataset(train_m, tr_m)
            ctx = train_f.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb = pd.concat([ctx, tr_f], ignore_index=True)
            cmb_fl = flatten_data(cmb, TARGET, N_LAGS)
            tr_fl = cmb_fl.tail(1)  # last row = test row
            X_te = np.hstack([sc.transform(tr_fl[sc_cols].values), tr_fl[us_cols].values]) if us_cols else sc.transform(tr_fl[sc_cols].values)
            predictions[vn].append(m.predict(X_te)[0])
        except Exception:
            predictions[vn].append(np.nan); failed += 1
    if (i+1)%8==0: print("  {} / {}".format(i+1, len(test_quarters)))
print("Done. {} failures.".format(failed))


C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

Done. 0 failures.


C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

In [5]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({"date": test_quarters, "actual": actuals_list,
                  "prediction": predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR, "ridge_{}.csv".format(vn)), index=False)
    print("Saved ridge_{}.csv".format(vn))


Saved ridge_m1.csv
Saved ridge_m2.csv
Saved ridge_m3.csv


In [6]:
def rmse(a,p):
    m=~(np.isnan(a)|np.isnan(p)); return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan
def mae(a,p):
    m=~(np.isnan(a)|np.isnan(p)); return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan
panels = {"Pre-COVID":"2017-01-01,2019-12-31","COVID":"2020-04-01,2021-12-31",
          "Post-COVID":"2022-01-01,2025-12-31","Full":"2017-01-01,2025-12-31"}
a=np.array(actuals_list); d=pd.to_datetime(test_quarters)
for pn, rng in panels.items():
    ps,pe=rng.split(","); m=(d>=ps)&(d<=pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp=np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}".format(vn,rmse(a[m],pp[m]),mae(a[m],pp[m])))


--- Pre-COVID ---
  m1  RMSFE=0.003052  MAE=0.002495
  m2  RMSFE=0.003072  MAE=0.002440
  m3  RMSFE=0.002861  MAE=0.002200
--- COVID ---
  m1  RMSFE=0.042509  MAE=0.026814
  m2  RMSFE=0.030596  MAE=0.019327
  m3  RMSFE=0.042624  MAE=0.026434
--- Post-COVID ---
  m1  RMSFE=0.005692  MAE=0.004244
  m2  RMSFE=0.005010  MAE=0.004039
  m3  RMSFE=0.004657  MAE=0.003435
--- Full ---
  m1  RMSFE=0.019483  MAE=0.008477
  m2  RMSFE=0.014367  MAE=0.006896
  m3  RMSFE=0.019419  MAE=0.007965
